# 第 2 讲 · Python 附录：用 pandas 走一遍同一条数据旅程

本笔记用 pandas 复刻讲义 `02_data.qmd` 的关键几步——读入、追加、合并、清洗、聚合、画图。思路与 Stata 版**完全一致**，只是命令换了语言。数据从课程 GitHub 仓库联网读取，无需克隆本仓库。

> 说明：本骨架的单元格输出待缓冲期补全并冻结；此处先给出可运行的代码与「翻译」提示词。

## 让 AI 把 Stata 流程翻译成 pandas

把讲义里的某段 Stata 代码（含 merge、缺失与离群处理、聚合）粘给 AI：

> 请翻译成等价的 pandas 代码，逐句对应并在注释里标出对应的 Stata 命令；保留合并匹配率与样本量检查；指出两种语言在缺失值、类型转换上的差异，哪些地方容易译错。

In [ ]:
import pandas as pd
import numpy as np

DATA = "https://raw.githubusercontent.com/lianxhcn/PXa2026a/main/examples/ch02/data"

fin1    = pd.read_stata(f"{DATA}/firm_finance_2021.dta")
fin2    = pd.read_stata(f"{DATA}/firm_finance_2022.dta")
lookup  = pd.read_stata(f"{DATA}/industry_lookup.dta")
monthly = pd.read_stata(f"{DATA}/mktcap_monthly.dta")
fin1.head()

In [ ]:
# 追加两批（≈ Stata 的 append）
panel = pd.concat([fin1, fin2], ignore_index=True)
# 主键唯一性（≈ isid stkcd year）
assert not panel.duplicated(["stkcd", "year"]).any()
print(panel.shape)                 # (126, 7)
panel["year"].value_counts()

In [ ]:
# 清洗合并键后左连接（≈ merge m:1 indcd，先清 "c36 " -> "C36"）
lookup["indcd"] = lookup["indcd"].str.strip().str.upper()
panel = panel.merge(lookup, on="indcd", how="left", indicator=True)
print(panel["_merge"].value_counts())   # 全部 both = 匹配率 100%
panel = panel.drop(columns="_merge")

In [ ]:
# 缺失：-99 还原为 NaN（≈ mvdecode rd, mv(-99)）
panel["rd"] = panel["rd"].replace(-99, np.nan)
# 离群：查明是录入错误，改正（≈ replace sales=... if ...）
mask = (panel["stkcd"] == "000101") & (panel["year"] == 2022)
panel.loc[mask, "sales"] = 108000
# 构造变量
panel["rd_ratio"] = 100 * panel["rd"] / panel["sales"]   # 研发强度(%)
panel["size"]     = np.log(panel["at"])                  # 企业规模

In [ ]:
# 频率转换/聚合：月度市值 -> 公司-年度（≈ collapse (mean) mktcap, by(stkcd year)）
monthly["year"] = monthly["ym"] // 100
mkt_year = monthly.groupby(["stkcd", "year"], as_index=False)["mktcap"].mean()
# 修键：数值 stkcd -> 6 位字符（补前导零，≈ tostring ... format(%06.0f)）
mkt_year["stkcd"] = mkt_year["stkcd"].astype(int).map(lambda x: f"{x:06d}")
mkt_year.head()

In [ ]:
import matplotlib.pyplot as plt
plt.rcParams["font.sans-serif"] = ["SimHei"]   # 中文显示（按本机字体调整）
plt.rcParams["axes.unicode_minus"] = False

ind_map = {"C39": "电子", "C27": "医药", "C36": "汽车"}
panel["ind_s"] = panel["indcd"].map(ind_map)
panel.boxplot(column="rd_ratio", by="ind_s")
plt.title("各行业研发强度"); plt.suptitle(""); plt.ylabel("研发强度(%)")
plt.show()

同一套判断——先看结构、查主键、清洗、构造、留痕——在 pandas 里同样成立。**语法交给 AI 翻译，判断留给你自己。**